# Notebook 02 — Analyse des features comportementales

Extraction des 16 features sur la fixture synthétique et analyse de leur distribution.
Objectif : valider que les features discriminent bien les fenêtres normales vs. spray.

In [ ]:
import sys
sys.path.insert(0, '../src')

import csv
from datetime import timedelta
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ueba.adapters.wazuh import WazuhAdapter
from ueba.domain.features import UEBAFeatureExtractor, FEATURE_NAMES

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Extraction des vecteurs de features

In [ ]:
FIXTURE = Path('../tests/integration/fixtures/sample_logs.csv')

with FIXTURE.open(newline='', encoding='utf-8') as f:
    records = list(csv.DictReader(f))

adapter = WazuhAdapter()
events = adapter.normalize(records)

extractor = UEBAFeatureExtractor(
    window_size=timedelta(hours=1),
    window_step=timedelta(minutes=30),
)
vectors = extractor.extract(events)

rows = []
for v in vectors:
    row = {'user': v.user, 'window_start': v.window_start, 'day': v.window_start.day}
    for name in FEATURE_NAMES:
        row[name] = getattr(v, name)
    rows.append(row)

df = pd.DataFrame(rows)
print(f'Vecteurs extraits : {len(df)}')
df.describe()

## 2. Comparaison Normal (13 mai) vs. Spray (16 mai)

In [ ]:
df['scenario'] = df['day'].map({13: 'Normal (13 mai)', 16: 'Spray (16 mai)'})

spray_features = ['failed_login_count', 'failed_login_ratio', 'login_count', 'off_hours_ratio']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flatten(), spray_features):
    sns.boxplot(data=df, x='scenario', y=feat, ax=ax, palette='Set2')
    ax.set_title(feat)
    ax.set_xlabel('')

fig.suptitle('Distribution des features : Normal vs. Password Spray', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Heatmap de corrélation des 16 features

In [ ]:
corr = df[list(FEATURE_NAMES)].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = corr.abs() < 0.1
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Corrélation entre les 16 features comportementales', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Timeline des failed_login_count par utilisateur (16 mai)

In [ ]:
spray_df = df[df['day'] == 16].copy()
spray_df['hour'] = spray_df['window_start'].dt.hour + spray_df['window_start'].dt.minute / 60

fig, ax = plt.subplots(figsize=(12, 5))
for user, grp in spray_df.groupby('user'):
    ax.plot(grp['hour'], grp['failed_login_count'], marker='o', label=user)

ax.axvline(14, color='red', linestyle='--', alpha=0.6, label='14h — début spray')
ax.set_xlabel('Heure de début de fenêtre (16 mai 2026)')
ax.set_ylabel('failed_login_count')
ax.set_title('Pic synchronisé d\'échecs de connexion — signature du Password Spray')
ax.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()